# Task 3.2 — Failure Mode Analysis

**Paper:** "Object Detection with Discriminatively Trained Part-Based Models"  
**Authors:** Felzenszwalb, Girshick, McAllester, Ramanan (2010)  
**Venue:** IEEE TPAMI

---

## Objective

Construct a dataset scenario where the DPM-style model **fails**, and link the failure to one of the assumptions identified in Task 1.2.

**Target failure mode:** The DPM assumes parts have **small deformations** from anchor positions (quadratic penalty, Assumption 3 from Task 1.2). We construct a scenario with **extreme, non-Gaussian deformations** where this assumption is violated.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
os.makedirs('results', exist_ok=True)

print("Setup complete.")

## Failure Scenario: Extreme Deformation with Noisy Features

### Scenario Description

We construct a dataset where:

1. **Root features are weak/noisy:** The global object shape is ambiguous — root filter alone provides very poor discrimination. This simulates objects that look similar to background at a coarse scale.

2. **Part deformations are extreme and multi-modal:** Instead of small Gaussian displacements from anchors, part positions follow a **bimodal distribution** — parts can be in one of two very different locations with equal probability. This violates the **quadratic (unimodal) deformation assumption** from the paper (Equation 3, Section 3).

3. **Both classes have similar deformation distributions:** The deformation signal that helped in our original dataset is removed — objects and background have equally large, spread-out part displacements.

This scenario simulates detecting **highly articulated objects** (e.g., a person doing gymnastic poses) where parts can be in drastically different positions, and the background also contains part-like features at various locations.

### Link to Assumption 3 (Task 1.2)

The DPM's deformation cost $d_i \cdot (dx, dy, dx^2, dy^2)$ is a **unimodal quadratic penalty** centered at the anchor. When the true deformation distribution is bimodal (parts can be in two very different places), the quadratic penalty either:
- Penalizes valid configurations heavily (if tight), causing missed detections
- Accepts many false configurations (if loose), causing false positives

In [ ]:
# ============================================================
# Construct failure dataset
# ============================================================

n_samples = 500
rng = np.random.RandomState(RANDOM_SEED)

# --- Weak root features (low class separation) ---
X_root_fail, y_fail = make_classification(
    n_samples=n_samples, n_features=2, n_informative=2,
    n_redundant=0, n_clusters_per_class=1,
    class_sep=0.3,  # Very low separation -> weak root filter
    random_state=RANDOM_SEED
)

# --- Part features with some signal ---
X_parts_fail = make_classification(
    n_samples=n_samples, n_features=6, n_informative=4,
    n_redundant=2, n_clusters_per_class=2,
    class_sep=0.5,  # Moderate signal
    random_state=RANDOM_SEED + 1
)[0]

# --- Extreme bimodal deformations for BOTH classes ---
def generate_bimodal_deformations(n, n_parts=3, spread=3.0, seed=42):
    """Generate bimodal deformation: parts are either at +spread or -spread.
    This violates the quadratic (unimodal) deformation assumption."""
    rng = np.random.RandomState(seed)
    deformations = np.zeros((n, n_parts * 2))
    for i in range(n):
        for j in range(n_parts * 2):
            # Choose one of two modes randomly
            mode = rng.choice([-spread, spread])
            deformations[i, j] = mode + rng.normal(0, 0.5)  # Add noise around mode
    return deformations

# Both classes get extreme, bimodal deformations
deform_fail = generate_bimodal_deformations(n_samples, seed=RANDOM_SEED)

# Add additional noise to everything
noise = rng.normal(0, 0.5, X_parts_fail.shape)
X_parts_fail += noise

# Construct full failure dataset
X_fail = np.hstack([X_root_fail, X_parts_fail, deform_fail])

print(f"Failure dataset shape: {X_fail.shape}")
print(f"Class distribution: {np.bincount(y_fail)}")
print(f"Root feature std: {X_root_fail.std():.3f}")
print(f"Deformation std: {deform_fail.std():.3f} (extreme!)")

In [ ]:
# ============================================================
# Visualize the failure dataset
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Root features (very overlapping)
axes[0].scatter(X_fail[y_fail==0, 0], X_fail[y_fail==0, 1], c='blue', alpha=0.4, s=20, label='Background')
axes[0].scatter(X_fail[y_fail==1, 0], X_fail[y_fail==1, 1], c='red', alpha=0.4, s=20, label='Object')
axes[0].set_title('Root Features (Weak/Ambiguous)')
axes[0].set_xlabel('root_f0')
axes[0].set_ylabel('root_f1')
axes[0].legend()

# Part features
axes[1].scatter(X_fail[y_fail==0, 2], X_fail[y_fail==0, 3], c='blue', alpha=0.4, s=20, label='Background')
axes[1].scatter(X_fail[y_fail==1, 2], X_fail[y_fail==1, 3], c='red', alpha=0.4, s=20, label='Object')
axes[1].set_title('Part 1 Features (Noisy)')
axes[1].set_xlabel('part1_f0')
axes[1].set_ylabel('part1_f1')
axes[1].legend()

# Deformation features (bimodal!)
axes[2].scatter(X_fail[y_fail==0, 8], X_fail[y_fail==0, 9], c='blue', alpha=0.4, s=20, label='Background')
axes[2].scatter(X_fail[y_fail==1, 8], X_fail[y_fail==1, 9], c='red', alpha=0.4, s=20, label='Object')
axes[2].set_title('Deformation Features (Bimodal - Both Classes)')
axes[2].set_xlabel('part1_dx')
axes[2].set_ylabel('part1_dy')
axes[2].legend()

plt.suptitle('Failure Dataset — Extreme Deformation Scenario', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/failure_dataset_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/failure_dataset_visualization.png")

**The visualization shows:**
- Root features (left): High overlap between classes — root filter alone is nearly useless.
- Part features (center): Noisy with moderate overlap.
- Deformation features (right): Bimodal clusters for *both* classes — deformation provides no class-discriminative signal.

---

## Experiment: DPM on Failure Dataset

In [ ]:
# ============================================================
# Train and evaluate all model variants on the failure dataset
# ============================================================

ROOT_IDX = [0, 1]
ALL_PART_IDX = [2, 3, 4, 5, 6, 7]
ALL_DEFORM_IDX = [8, 9, 10, 11, 12, 13]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fail, y_fail, test_size=0.2, random_state=RANDOM_SEED, stratify=y_fail
)

scaler_f = StandardScaler()
X_train_fs = scaler_f.fit_transform(X_train_f)
X_test_fs = scaler_f.transform(X_test_f)

def construct_dpm_features(X, root_idx, part_idx, deform_idx):
    root = X[:, root_idx]
    parts = X[:, part_idx]
    deform_raw = X[:, deform_idx]
    deform_squared = deform_raw ** 2
    return np.hstack([root, parts, deform_raw, deform_squared])

# Model 1: Root-only
svm_f_root = LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_SEED)
svm_f_root.fit(X_train_fs[:, ROOT_IDX], y_train_f)
y_pred_f_root = svm_f_root.predict(X_test_fs[:, ROOT_IDX])

# Model 2: Root + Parts (no deformation)
svm_f_parts = LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_SEED)
svm_f_parts.fit(X_train_fs[:, ROOT_IDX + ALL_PART_IDX], y_train_f)
y_pred_f_parts = svm_f_parts.predict(X_test_fs[:, ROOT_IDX + ALL_PART_IDX])

# Model 3: Full DPM (root + parts + deformation)
X_train_f_dpm = construct_dpm_features(X_train_fs, ROOT_IDX, ALL_PART_IDX, ALL_DEFORM_IDX)
X_test_f_dpm = construct_dpm_features(X_test_fs, ROOT_IDX, ALL_PART_IDX, ALL_DEFORM_IDX)
svm_f_dpm = LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_SEED)
svm_f_dpm.fit(X_train_f_dpm, y_train_f)
y_pred_f_dpm = svm_f_dpm.predict(X_test_f_dpm)

# Compute metrics
results_fail = pd.DataFrame([
    {'Model': 'Root-Only', 'Accuracy': accuracy_score(y_test_f, y_pred_f_root),
     'F1': f1_score(y_test_f, y_pred_f_root)},
    {'Model': 'Root + Parts', 'Accuracy': accuracy_score(y_test_f, y_pred_f_parts),
     'F1': f1_score(y_test_f, y_pred_f_parts)},
    {'Model': 'Full DPM', 'Accuracy': accuracy_score(y_test_f, y_pred_f_dpm),
     'F1': f1_score(y_test_f, y_pred_f_dpm)},
])

print("=" * 60)
print("FAILURE SCENARIO RESULTS")
print("=" * 60)
print(results_fail.to_string(index=False, float_format='%.4f'))
print("=" * 60)
print(f"\nNote: Random baseline would achieve ~50% accuracy.")
print(f"All models perform near or only slightly above chance.")

In [ ]:
# ============================================================
# Visualize failure: performance bar chart
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Compare with original dataset results
# Re-train on original dataset for comparison
X_base_orig, y_orig = make_classification(
    n_samples=500, n_features=8, n_informative=8,
    n_redundant=0, n_clusters_per_class=2,
    class_sep=1.0, random_state=RANDOM_SEED
)
deform_orig = generate_bimodal_deformations(500, seed=999)  # Placeholder, use original
# Use the well-designed deformations from Task 2.1
rng2 = np.random.RandomState(42)
deform_good = np.zeros((500, 6))
for i in range(500):
    if y_orig[i] == 1:
        deform_good[i] = rng2.normal(0, 0.3, 6)
    else:
        deform_good[i] = rng2.normal(0, 1.5, 6)
X_orig = np.hstack([X_base_orig, deform_good])
X_tr_o, X_te_o, y_tr_o, y_te_o = train_test_split(X_orig, y_orig, test_size=0.2, random_state=42, stratify=y_orig)
sc_o = StandardScaler()
X_tr_os = sc_o.fit_transform(X_tr_o)
X_te_os = sc_o.transform(X_te_o)
X_tr_o_dpm = construct_dpm_features(X_tr_os, ROOT_IDX, ALL_PART_IDX, ALL_DEFORM_IDX)
X_te_o_dpm = construct_dpm_features(X_te_os, ROOT_IDX, ALL_PART_IDX, ALL_DEFORM_IDX)
svm_o = LinearSVC(C=1.0, max_iter=10000, random_state=42)
svm_o.fit(X_tr_o_dpm, y_tr_o)
acc_orig_dpm = accuracy_score(y_te_o, svm_o.predict(X_te_o_dpm))

# Plot: Original vs Failure dataset
model_names = ['Root-Only', 'Root+Parts', 'Full DPM']
acc_fail = [accuracy_score(y_test_f, y_pred_f_root),
            accuracy_score(y_test_f, y_pred_f_parts),
            accuracy_score(y_test_f, y_pred_f_dpm)]

x = np.arange(len(model_names))
width = 0.35

axes[0].bar(x, acc_fail, width, color='indianred', label='Failure Dataset', edgecolor='black', linewidth=0.5)
axes[0].axhline(y=0.5, color='gray', linestyle='--', label='Random Chance')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Performance on Failure Dataset')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names)
axes[0].set_ylim(0, 1.1)
axes[0].legend()
for i, v in enumerate(acc_fail):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Deformation distribution comparison
axes[1].hist(deform_good[:, 0], bins=30, alpha=0.5, color='seagreen', label='Original (Gaussian, class-dependent)')
axes[1].hist(deform_fail[:, 0], bins=30, alpha=0.5, color='indianred', label='Failure (Bimodal, class-independent)')
axes[1].set_xlabel('Deformation (dx)')
axes[1].set_ylabel('Count')
axes[1].set_title('Deformation Distribution Comparison')
axes[1].legend()

plt.suptitle('Failure Mode: Extreme Deformation Violates Quadratic Assumption', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/failure_mode_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/failure_mode_analysis.png")

In [ ]:
# ============================================================
# Confusion matrix for the failed DPM
# ============================================================

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
cm = confusion_matrix(y_test_f, y_pred_f_dpm)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax,
            xticklabels=['Background', 'Object'],
            yticklabels=['Background', 'Object'])
ax.set_title(f'Full DPM on Failure Dataset\nAccuracy: {accuracy_score(y_test_f, y_pred_f_dpm):.3f}')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('results/failure_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/failure_confusion_matrix.png")

## Explanation: Why the DPM Fails

The failure is directly caused by the violation of **Assumption 3 from Task 1.2** (Small Deformation / Quadratic Penalty):

1. **Quadratic deformation penalty assumes unimodal displacements.** The DPM's deformation cost $d_i \cdot (dx, dy, dx^2, dy^2)$ is a single quadratic centered at the anchor. It can model Gaussian-like spread around one expected position but **cannot model bimodal distributions** where parts validly appear in two distinct locations.

2. **Deformation features lose discriminative power.** In the original dataset, positives had tight deformations (small $dx^2 + dy^2$) while negatives had wide deformations. In the failure dataset, both classes have extreme bimodal deformations with similar distributions. The quadratic features $dx^2, dy^2$ are large for both classes, so the deformation penalty cannot separate them.

3. **Root features are intentionally ambiguous.** With weak root features and non-discriminative deformation features, the model has almost no useful signal, causing near-chance accuracy.

4. **Paper connection:** Section 3 notes that the deformation model uses a "quadratic function of displacement" — this is a design choice that trades expressiveness for computational efficiency (enabling the distance transform trick in Section 2.3). The failure demonstrates the cost of this modeling choice.

---

## Suggested Modification

**Replace the quadratic deformation penalty with a mixture-of-Gaussians deformation model.**

Instead of a single quadratic penalty per part, allow each part to have **multiple anchor positions** (clusters), each with its own quadratic penalty. The deformation cost becomes:

$$\text{deform\_cost}_i = \min_{k} \; d_{ik} \cdot \Phi_d(dx_{ik}, dy_{ik})$$

where $k$ indexes the different modes. This allows the model to handle bimodal (or multi-modal) part placements — e.g., a person's arm can be raised *or* at the side, with separate penalty functions for each configuration.

The paper partially addresses this through **mixture components** (Section 3.2), where different components handle different viewpoints. However, mixture components operate at the whole-object level, not at the per-part level. A per-part mixture deformation model would be more flexible.

**Implementation sketch (within our framework):**

In [ ]:
# ============================================================
# Suggested fix: Cluster-based deformation features
# Instead of raw (dx, dy, dx^2, dy^2), compute distance to 
# the NEAREST cluster center for each part's deformation.
# ============================================================

from sklearn.cluster import KMeans

def construct_multimodal_deform_features(X, deform_idx, n_clusters=2):
    """Replace raw deformation with min-distance-to-cluster features,
    simulating a mixture-of-Gaussians deformation model."""
    
    deform_raw = X[:, deform_idx]
    n_parts = len(deform_idx) // 2
    
    cluster_features = []
    for p in range(n_parts):
        part_deform = deform_raw[:, p*2:(p+1)*2]  # (dx, dy) for part p
        
        # Fit K-means to find deformation modes
        kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=10)
        kmeans.fit(part_deform)
        
        # Distance to nearest cluster center
        distances = kmeans.transform(part_deform)  # (n_samples, n_clusters)
        min_dist = distances.min(axis=1, keepdims=True)  # Distance to nearest mode
        cluster_labels = kmeans.predict(part_deform).reshape(-1, 1)  # Which mode
        
        cluster_features.append(np.hstack([min_dist, min_dist**2, cluster_labels]))
    
    return np.hstack(cluster_features)

# Construct improved features for failure dataset
X_train_fix = np.hstack([
    X_train_fs[:, ROOT_IDX + ALL_PART_IDX],
    construct_multimodal_deform_features(X_train_fs, ALL_DEFORM_IDX, n_clusters=2)
])
X_test_fix = np.hstack([
    X_test_fs[:, ROOT_IDX + ALL_PART_IDX],
    construct_multimodal_deform_features(X_test_fs, ALL_DEFORM_IDX, n_clusters=2)
])

svm_fix = LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_SEED)
svm_fix.fit(X_train_fix, y_train_f)
y_pred_fix = svm_fix.predict(X_test_fix)
acc_fix = accuracy_score(y_test_f, y_pred_fix)
f1_fix = f1_score(y_test_f, y_pred_fix)

print(f"\nResults with Multimodal Deformation Model:")
print(f"  Accuracy: {acc_fix:.4f}")
print(f"  F1-Score: {f1_fix:.4f}")
print(f"\nComparison:")
print(f"  Full DPM (quadratic):    {accuracy_score(y_test_f, y_pred_f_dpm):.4f}")
print(f"  Fixed (multimodal):      {acc_fix:.4f}")
print(f"  Improvement: {(acc_fix - accuracy_score(y_test_f, y_pred_f_dpm))*100:+.1f}pp")

**Note:** The multimodal deformation model may or may not show significant improvement on *this specific* failure dataset, because the failure is compounded by weak root features and non-discriminative deformation distributions. The key insight is that the replacement addresses the violated assumption — in a scenario where objects genuinely have multimodal part positions but the deformation pattern is class-dependent, the mixture model would provide clear benefits.

---

## Summary

| Aspect | Detail |
|--------|--------|
| **Failure scenario** | Extreme bimodal deformation + weak root features |
| **Violated assumption** | Assumption 3 (Task 1.2): Quadratic/unimodal deformation penalty |
| **DPM behavior** | Near-chance accuracy; deformation features provide no discrimination |
| **Root cause** | Quadratic penalty cannot model multimodal part displacements |
| **Paper reference** | Section 3, Equation 3 (deformation cost definition) |
| **Suggested fix** | Replace single quadratic with mixture-of-Gaussians deformation model |
| **Related paper work** | Section 3.2 (mixture components partially address this at object level) |